In [ ]:
!pip install tqdm

In [ ]:
# @title Manga Upscaler + Downscaler (Per-Image Timer) { display-mode: "form" }

import os
from google.colab import drive
import subprocess
from pathlib import Path
import torch
from PIL import Image
import shutil
import time

# ==== ✅ CUSTOM FOLDERS ====
BW_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN (1)/bw/Vol.01 Ch.0002 - Rope Partner (en) [Bakana Haven]"
COLOR_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN/color"
FINAL_OUTPUT_FOLDER = "/content/gdr"
# ==== CORE SETUP ====
def check_connect_gdrive():
    if not os.path.exists("/content/gdrive/MyDrive"):
        print("🔌 Mounting Google Drive...")
        drive.mount("/content/gdrive")

def check_clone_esrgan():
    if not os.path.exists("ESRGAN"):
        print("⬇️ Cloning ESRGAN...")
        subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"])

def init_dirs():
    Path(FINAL_OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    Path("/content/temp_input").mkdir(parents=True, exist_ok=True)
    Path("/content/temp_output").mkdir(parents=True, exist_ok=True)

def dir_contains_files(path):
    return os.path.exists(path) and any(Path(path).glob("*"))

def upscale_and_downscale_image(image_path, model_path):
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"

    # Clear temp folders
    for folder in [temp_input, temp_output]:
        for file in Path(folder).glob("*"):
            file.unlink()

    # Copy image to temp input
    shutil.copy(image_path, temp_input)

    # Run ESRGAN on the image
    subprocess.run([
        "python", "ESRGAN/upscale.py", "-se",
        "-i", temp_input,
        "-o", temp_output,
        model_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Resize immediately
    filename = os.path.basename(image_path)
    upscaled_path = os.path.join(temp_output, filename)
    final_output_path = os.path.join(FINAL_OUTPUT_FOLDER, filename)

    if os.path.exists(upscaled_path):
        with Image.open(upscaled_path) as img:
            new_size = (img.width // 2, img.height // 2)
            resized = img.resize(new_size, Image.LANCZOS)
            resized.save(final_output_path)
            return True
    return False

def process_folder(folder_path, model_path):
    print(f"\n📁 Processing: {folder_path}")
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    images = [f for f in os.listdir(folder_path) if Path(f).suffix.lower() in exts]

    total = len(images)
    for idx, file in enumerate(images, 1):
        image_path = os.path.join(folder_path, file)
        start = time.time()

        success = upscale_and_downscale_image(image_path, model_path)

        end = time.time()
        duration = end - start
        if success:
            print(f"✅ {file} ({idx}/{total}) | ⏱ {duration:.2f} sec")
        else:
            print(f"❌ Failed: {file} ({idx}/{total})")

def main():
    print("📈 Manga Upscaler + 2x Downscaler with Per-Image Timing")

    if not torch.cuda.is_available():
        print("❌ GPU not enabled. Set 'Runtime' → 'Change runtime type' → GPU.")
        return

    check_connect_gdrive()
    check_clone_esrgan()
    init_dirs()

    if dir_contains_files(BW_INPUT_FOLDER):
        print("🎨 Upscaling B&W images...")
        process_folder(BW_INPUT_FOLDER, "ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth")

    if dir_contains_files(COLOR_INPUT_FOLDER):
        print("🎨 Upscaling Color images...")
        process_folder(COLOR_INPUT_FOLDER, "ESRGAN/models/4x-AnimeSharp.pth")

    print(f"\n📁 Done. Final output saved to:\n{FINAL_OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')